# GovBench-Med — Experiment Runner
**Before anything:** Runtime → Change runtime type → **T4 GPU** → Save

Then run cells **one at a time**, top to bottom. Wait for each to finish (no ✗ errors) before moving on.

In [ ]:
# ── CELL 1: Confirm GPU ──
import subprocess
r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if r.returncode != 0:
    raise RuntimeError('NO GPU. Go to Runtime → Change runtime type → T4 GPU → Save, then reconnect.')
# Print just the GPU name line
for line in r.stdout.splitlines():
    if 'Tesla' in line or 'T4' in line or 'A100' in line or 'V100' in line:
        print('GPU:', line.strip())
        break
else:
    print(r.stdout[:300])
print('GPU confirmed.')

In [ ]:
# ── CELL 2: Install zstd + Ollama + start server + pull model ──
import subprocess, threading, time, urllib.request

# Step 1: install zstd (Ollama installer needs it)
print('Installing zstd...')
subprocess.run(['apt-get', 'install', '-y', '-q', 'zstd'], check=True)
print('  done')

# Step 2: install Ollama
print('Installing Ollama...')
r = subprocess.run('curl -fsSL https://ollama.com/install.sh | sh',
                   shell=True, capture_output=True, text=True)
if r.returncode != 0:
    print('STDERR:', r.stderr[-500:])
    raise RuntimeError('Ollama install failed')
print('  done')

# Step 3: start Ollama server
print('Starting Ollama server...')
def _serve():
    subprocess.run(['ollama', 'serve'],
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
threading.Thread(target=_serve, daemon=True).start()

# Step 4: wait until server responds
for i in range(40):
    time.sleep(2)
    try:
        urllib.request.urlopen('http://localhost:11434/api/tags', timeout=3)
        print(f'  server ready after {(i+1)*2}s')
        break
    except Exception:
        print(f'  waiting... {(i+1)*2}s', end='\r')
else:
    raise RuntimeError('Ollama server did not start. Re-run this cell.')

# Step 5: pull llama3.1:8b only (we run one model first to keep it simple)
print('Pulling llama3.1:8b (this takes 3-5 min on T4)...')
r = subprocess.run(['ollama', 'pull', 'llama3.1:8b'],
                   capture_output=True, text=True, timeout=900)
if r.returncode != 0:
    print('STDERR:', r.stderr[-300:])
    raise RuntimeError('Model pull failed')
print('  llama3.1:8b ready')

# Step 6: verify model is listed
r2 = subprocess.run(['ollama', 'list'], capture_output=True, text=True)
print(r2.stdout)

In [ ]:
# ── CELL 3: Speed test — confirm GPU inference is fast ──
import requests, time

print('Running speed test...')
t0 = time.time()
r = requests.post(
    'http://localhost:11434/api/generate',
    json={'model': 'llama3.1:8b', 'prompt': 'Say hello in 5 words.', 'stream': False},
    timeout=300
)
elapsed = time.time() - t0
print(f'Latency: {elapsed:.1f}s')
print(f'Response: {r.json()["response"][:80]}')

if elapsed < 20:
    print('GPU is working. Proceed to Cell 4.')
elif elapsed < 60:
    print('Slow but usable (might be CPU). Proceed carefully.')
else:
    print('TOO SLOW — you are on CPU. Go to Runtime → Change runtime type → T4 GPU.')

In [ ]:
# ── CELL 4: Clone repo + install deps ──
import subprocess, sys, os

if not os.path.exists('govbench-med'):
    print('Cloning repo...')
    subprocess.run(['git', 'clone',
        'https://github.com/nikki-nooka/govbench-med.git'], check=True)
else:
    print('Repo already cloned, pulling latest...')
    subprocess.run(['git', '-C', 'govbench-med', 'pull'])

os.chdir('govbench-med')
print('Working dir:', os.getcwd())

print('Installing Python deps...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'requests', 'pandas', 'numpy', 'matplotlib',
    'seaborn', 'scipy', 'scikit-learn', 'tqdm'], check=True)
print('Done.')

In [ ]:
# ── CELL 5: Download datasets ──
import subprocess, sys
r = subprocess.run([sys.executable, 'scripts/prepare_data.py'],
                   capture_output=True, text=True, timeout=300)
print(r.stdout)
if r.returncode != 0:
    print('ERR:', r.stderr[-500:])
    raise RuntimeError('prepare_data.py failed')

In [ ]:
# ── CELL 6: 5-case rehearsal — confirms full pipeline works ──
# Runs 5 cases × G0 × llama3.1:8b × seed=42
# Every case should show tokens > 0 and a real diagnosis.
# If you see 0tok or ABSTAIN here → do NOT run the full experiment yet.

import sys, json, time, requests
sys.path.insert(0, '.')

# Re-import after chdir
from src.governance.levels import run_governance
from src.evaluation.metrics import CaseEvaluator

with open('data/processed/cases.json') as f:
    cases = json.load(f)

# Use first 5 cases only
test_cases = cases[:5]
ha_ids  = {c['id'] for c in cases if c.get('is_high_acuity')}
amb_ids = {c['id'] for c in cases if c.get('is_ambiguous')}
evaluator = CaseEvaluator(ha_ids, amb_ids)

print('Running 5-case rehearsal...\n')
results = []
all_ok = True

for i, case in enumerate(test_cases):
    print(f'[{i+1}/5] {case["id"]}', end=' ... ', flush=True)
    t0 = time.time()
    gov = run_governance('G0', case, 'llama3.1:8b', 42)
    scored = evaluator.score(gov, case['ground_truth'])
    elapsed = time.time() - t0

    ok = gov.total_tokens > 0 and gov.top_diagnosis is not None
    status = '✓' if ok else '✗ FAILED'
    if not ok:
        all_ok = False

    print(f'{status} | {gov.total_tokens}tok | {elapsed:.1f}s | diag={gov.top_diagnosis}')
    results.append({
        'id': case['id'],
        'ok': ok,
        'tokens': gov.total_tokens,
        'latency': round(elapsed, 1),
        'diagnosis': gov.top_diagnosis,
        'ground_truth': case['ground_truth'],
    })

print()
passed = sum(r['ok'] for r in results)
avg_lat = sum(r['latency'] for r in results) / len(results)
print(f'Passed: {passed}/5')
print(f'Avg latency: {avg_lat:.1f}s/case')
print(f'Projected time for 50 cases × G0+G1+G2: ~{50*3*avg_lat/60:.0f} min')

if all_ok:
    print('\nAll 5 cases OK. Proceed to Cell 7 (full pilot).')
else:
    print('\nSome cases FAILED. Do not proceed. Check Ollama is still running:')
    print('  import subprocess; subprocess.run(["ollama", "list"])')

In [ ]:
# ── CELL 7: Full pilot — 50 cases × G0, G1, G2 × llama3.1:8b × seed=42 ──
# Only run this after Cell 6 shows 5/5 passed.
# This is the core experiment for your paper.
# Expected time on T4 GPU: ~30-45 min

import subprocess, sys

r = subprocess.run([
    sys.executable, 'scripts/run_experiments.py',
    '--n', '50',
    '--level', 'G0',
    '--level', 'G1',
    '--level', 'G2',
    '--model', 'llama3.1:8b',
    '--seed', '42'
], capture_output=False, timeout=7200)

print('Exit code:', r.returncode)
if r.returncode == 0:
    print('Pilot complete. Proceed to Cell 8 for results.')
else:
    print('Something failed — check output above.')

In [ ]:
# ── CELL 8: Results summary + Pareto curve ──
import pandas as pd, matplotlib.pyplot as plt, seaborn as sns, glob, os

csvs = sorted(glob.glob('experiments/results/results_*.csv'))
assert csvs, 'No results CSV found. Run Cell 7 first.'
df = pd.read_csv(csvs[-1])
print(f'Loaded {len(df)} rows')
print(df[['governance_level','model','top1_correct','critical_miss',
          'hallucination_impactful','total_tokens','total_latency']].head(10))

# Aggregate
agg = df.groupby('governance_level').agg(
    accuracy   = ('top1_correct', 'mean'),
    cmr        = ('critical_miss', 'mean'),
    hir        = ('hallucination_impactful', 'mean'),
    urr        = ('unsafe_reassurance', 'mean'),
    avg_tokens = ('total_tokens', 'mean'),
    avg_lat    = ('total_latency', 'mean'),
).reset_index()
agg['css'] = 1 - (0.5*agg['cmr'] + 0.3*agg['hir'] + 0.2*agg['urr'])
g0_tok = agg[agg['governance_level']=='G0']['avg_tokens'].values[0]
g0_lat = agg[agg['governance_level']=='G0']['avg_lat'].values[0]
agg['ccs'] = 0.6*(agg['avg_tokens']/g0_tok) + 0.4*(agg['avg_lat']/g0_lat)

print('\n--- Aggregate Results ---')
print(agg[['governance_level','css','cmr','hir','urr','accuracy','ccs']].to_string(index=False))

# Pareto curve
os.makedirs('paper/figures', exist_ok=True)
fig, ax = plt.subplots(figsize=(8,5))
ax.plot(agg['ccs'], agg['css'], 'o-', color='#2196F3', linewidth=2.5, markersize=10)
for _, row in agg.iterrows():
    ax.annotate(row['governance_level'], (row['ccs'], row['css']),
                textcoords='offset points', xytext=(8,4), fontsize=11, fontweight='bold')
ax.set_xlabel('Composite Cost Score (CCS) — normalised to G0', fontsize=12)
ax.set_ylabel('Clinical Safety Score (CSS)', fontsize=12)
ax.set_title('GovBench-Med: Governance–Cost Pareto Frontier\n(llama3.1:8b, 50 cases)', fontsize=13)
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig('paper/figures/pareto_frontier.png', dpi=300)
plt.show()
print('Saved to paper/figures/pareto_frontier.png')

In [ ]:
# ── CELL 9: Governance Efficiency (GE) table — your key result ──
print(f'{"Transition":<12} {"ΔCSS":>8} {"ΔCCS":>8} {"GE":>8}  Note')
print('-' * 55)
levels = agg.set_index('governance_level')
order  = [l for l in ['G0','G1','G2','G3','G4'] if l in levels.index]
for i in range(len(order)-1):
    l0, l1 = order[i], order[i+1]
    dcss = levels.loc[l1,'css'] - levels.loc[l0,'css']
    dccs = levels.loc[l1,'ccs'] - levels.loc[l0,'ccs']
    ge   = dcss / dccs if abs(dccs) > 1e-6 else float('inf')
    note = 'efficient' if ge >= 1 else ('knee of curve' if 0 < ge < 1 else 'no gain')
    print(f'{l0}→{l1:<8} {dcss:>8.4f} {dccs:>8.4f} {ge:>8.4f}  ← {note}')

In [ ]:
# ── CELL 10: Download results ──
from google.colab import files
import glob, zipfile

with zipfile.ZipFile('/content/govbench_results.zip', 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in glob.glob('experiments/results/*.csv'):
        zf.write(f)
    for f in glob.glob('experiments/results/*.json'):
        zf.write(f)
    for f in glob.glob('paper/figures/*.png'):
        zf.write(f)

print('Downloading...')
files.download('/content/govbench_results.zip')